In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os

def train_test_file(root, dir):
    file_txt = open(dir + '.txt', 'w')  # 打开一个文件准备写入训练或测试集信息
    path = os.path.join(root, dir)  # 拼接成完整的路径
    print(f"Processing directory: {path}")  # 调试信息
    for roots, directories, files in os.walk(path):  # 遍历目录及其子目录
        print(f"Roots: {roots}, Directories: {directories}, Files: {files}")  # 调试信息
        if directories:
            dirs = directories  # 如果有子目录，保存子目录列表
            print(f"Directories found: {dirs}")  # 调试信息
        else:
            now_dir = roots.split('/')  # 修改路径分隔符为正斜杠
            for file in files:
                path_1 = os.path.join(roots, file)  # 获取每个文件的完整路径
                print(f"File path: {path_1}")  # 调试信息
                file_txt.write(path_1 + ' ' + str(dirs.index(now_dir[-1])) + '\n')  # 写入文件路径和标签
    file_txt.close()  # 关闭文件

# 设置详细路径
root = r'/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/食物分类/food_dataset2'
train_dir = 'train'
test_dir = 'test'

# 生成训练集和测试集文件
train_test_file(root, train_dir)  # 生成训练集文件
train_test_file(root, test_dir)   # 生成测试集文件

# 检查输出文件内容
train_file = '/content/train.txt'
test_file = '/content/test.txt'

print("train.txt 内容:")
with open(train_file, 'r') as file:
    print(file.read())

print("test.txt 内容:")
with open(test_file, 'r') as file:
    print(file.read())

流式输出内容被截断，只能显示最后 5000 行内容。
/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/食物分类/food_dataset2/train/厨余垃圾_鸡翅/img_鸡翅_256.jpeg 1
/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/食物分类/food_dataset2/train/厨余垃圾_鸡翅/img_鸡翅_262.jpeg 1
/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/食物分类/food_dataset2/train/厨余垃圾_鸡翅/img_鸡翅_266.jpeg 1
/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/食物分类/food_dataset2/train/厨余垃圾_鸡翅/img_鸡翅_264.jpeg 1
/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/食物分类/food_dataset2/train/厨余垃圾_鸡翅/img_鸡翅_265.jpeg 1
/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/食物分类/food_dataset2/train/厨余垃圾_鸡翅/img_鸡翅_267.jpeg 1
/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/食物分类/food_dataset2/train/厨余垃圾_鸡翅/img_鸡翅_263.jpeg 1
/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/食物分类/food_dataset2/train/厨余垃圾_鸡翅/img_鸡翅_27.jpeg 1
/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/食物分类/food_dataset2/train/厨余垃圾_鸡翅/img_鸡翅_268.jpeg 1
/content/drive/My Drive/Python人工智能PyTorch教程素材/第3章/食物分类/food_datas

In [ ]:
# 迁移学习.py
import torch
import torchvision.models as models
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np


'''将模型迁移到食物分类项目中'''
resnet_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
#weights=models.ResNet18_Weights.DEFAULT表示使用在 ImageNet 数据集上预先训练好的权重来初始化模型参数
# 如果将下面俩行注释代表是全模型训练（当然还括params_to_update = [] 及下面三行。）
for param in resnet_model.parameters():
            param.requires_grad = False #uires_grad = False)，只训练最后的全连接层
#模型所有参数（即权重和偏差）的requires_grad属性设置为False，从而冻结所有模型参数，
# 使得在反向传播过程中不会计算它们的梯度，以此减少模型的计算量，提高推理速度。
in_features = resnet_model.fc.in_features       #获取模型原输入的特征个数
resnet_model.fc = nn.Linear(in_features, 20)    #创建一个全连接层，输入特征为in_features，输出为20
#  (假设有 20 种食物类别)。
# 如果将下面四行注释代表是全模型训练
params_to_update = []   #保存需要训练的参数，仅仅包含全连接层的参数
for param in resnet_model.parameters():
    if param.requires_grad == True:
        params_to_update.append(param)

data_transforms = {
    'train':
        transforms.Compose([
        transforms.Resize([300,300]),   #是图像变换大小

        transforms.RandomRotation(45),#随机旋转，-45到45度之间随机选
        transforms.CenterCrop(224),#从中心开始裁剪
        transforms.RandomHorizontalFlip(p=0.5),#随机水平翻转 选择一个概率概率
        transforms.RandomVerticalFlip(p=0.5),#随机垂直翻转
        # transforms.ColorJitter(brightness=0.2, contrast=0.1, saturation=0.1, hue=0.1),#参数1为亮度，参数2为对比度，参数3为饱和度，参数4为色相
        transforms.RandomGrayscale(p=0.1),#概率转换成灰度率，3通道就是R=G=B
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])#归一化，均值，标准差
    ]),
    'valid':
        transforms.Compose([
        transforms.Resize([224,224]),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

class food_dataset(Dataset):    #food_dataset是自己创建的类名称，可以改为你需要的名称
    def __init__(self, file_path,transform=None): #类的初始化
        self.file_path = file_path
        self.imgs = []
        self.labels = []
        self.transform = transform
        with open(self.file_path, encoding='utf-8') as f:  # 指定编码为 utf-8
            samples = [x.strip().rsplit(' ', 1) for x in f.readlines()] # 使用 rsplit 并指定最大分割次数为 1
            for img_path, label in samples:
                self.imgs.append(img_path)
                self.labels.append(label)

    # ... 其余代码保持不变 ...

    def __len__(self):  #类实例化对象后，可以使用len函数测量对象的个数，方法返回数据集的大小。
        return len(self.imgs)

    def __getitem__(self, idx): #关键，可通过索引的形式获取每一个图片数据及标签，方法根据索引返回对应的图像和标签，并应用预处理。
        image = Image.open(self.imgs[idx])   #
        if self.transform:
            image = self.transform(image)

        label = self.labels[idx]
        label = torch.tensor(int(label), dtype=torch.long)  # 将标签转换为 torch.LongTensor 类型)
        return image, label

# 使用 food_dataset 类创建训练集 (training_data) 和测试集 (test_data)。
# 使用 DataLoader 类创建数据加载器 (train_dataloader, test_dataloader)，用于将数据分成批次加载。
# training_data = food_dataset(file_path = './train.txt',transform = data_transforms['train'])
# test_data = food_dataset(file_path = './test.txt',transform = data_transforms['valid'])
training_data = food_dataset(file_path = '/content/train.txt',transform = data_transforms['train'])
test_data = food_dataset(file_path = '/content/test.txt',transform = data_transforms['valid'])

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using {device} device")

model = resnet_model.to(device) #

loss_fn = nn.CrossEntropyLoss() #创建交叉熵损失函数对象，因为手写字识别中一共有10个数字，输出会有10个结果
optimizer = torch.optim.Adam(resnet_model.parameters(), lr=0.001)#仅训练最后一层参数
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)#训练全部参数



def train(dataloader, model, loss_fn, optimizer):
    model.train()
    for X, y in dataloader:                 #其中batch为每一个数据的编号
        X, y = X.to(device), y.to(device)   #把训练数据集和标签传入cpu或GPU
        pred = model.forward(X)             #自动初始化 w权值
        loss = loss_fn(pred, y)             #通过交叉熵损失函数计算损失值loss
        optimizer.zero_grad()               #梯度值清零
        loss.backward()                     #反向传播计算得到每个参数的梯度值
        optimizer.step()                    #根据梯度更新网络参数

best_acc = 0
def test(dataloader, model, loss_fn):
    global best_acc
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()    #
    test_loss, correct = 0, 0
    with torch.no_grad():   #一个上下文管理器，关闭梯度计算。当你确认不会调用Tensor.backward()的时候。这可以减少计算所用内存消耗。
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model.forward(X)
            test_loss += loss_fn(pred, y).item() #
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test result: \n Accuracy: {(100*correct)}%, Avg loss: {test_loss}")
    acc_s.append(correct)
    loss_s.append(test_loss)

    if correct > best_acc:
        best_acc = correct


'''训练模型'''
epochs = 5
acc_s = []
loss_s = []
for t in range(epochs):
    train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)  # 64张图片为一个包，
    test_dataloader = DataLoader(test_data, batch_size=64, shuffle=True)
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print('最优训练结果为：',best_acc)

Using cpu device
Epoch 1
-------------------------------
Test result: 
 Accuracy: 28.333333333333332%, Avg loss: 3.0733734369277954
Epoch 2
-------------------------------
Test result: 
 Accuracy: 27.500000000000004%, Avg loss: 3.5933552980422974
Epoch 3
-------------------------------
Test result: 
 Accuracy: 29.166666666666668%, Avg loss: 3.9746443033218384
Epoch 4
-------------------------------
Test result: 
 Accuracy: 27.500000000000004%, Avg loss: 4.292797088623047
Epoch 5
-------------------------------
Test result: 
 Accuracy: 27.500000000000004%, Avg loss: 4.558351755142212
最优训练结果为： 0.2916666666666667
